# Orquestador — clasificación de potencial por vertical

Corre las etapas 1 a 6 (datos → features → Equifax → modelo → decisión → ensamblado y export) para **una vertical de cliente** (todas menos `pm_mercado`, que tiene su propio flujo de agregación por mercado — ver `notebooks/orquestador_pm.ipynb`).

Para correr otra vertical, cambia `VERTICAL` en la siguiente celda y ejecuta todo el notebook. Para correrlas todas, usa la última sección.

In [ ]:
VERTICAL = "lavanderias"  # lavanderias | hoteles | panaderia | gastro | bodegas

# Rutas de los insumos locales (ajusta a tu entorno)
RUTA_LOCALES = "data_potential/input/vs2/locales.csv"
RUTA_EQUIFAX = "data_potential/input/vs2/equifax.csv"
RUTA_URBANICIDAD = "data_potential/input/vs2/urbanicidad.csv"
RUTA_SALIDA = "data_potential/output"
RUTA_METRICAS = "data_potential/output/metricas_modelos.csv"


In [ ]:
import sys, time, warnings
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))  # para importar `src` desde notebooks/

from src.config import get_vertical
from src.data import fetch_clientes, fetch_puntos
from src.features import agregar_features_cliente
from src.equifax import prepare_equifax_sum
from src.model import fit_potential, validation_table
from src.metrics_log import log_metrics
from src.decision import apply_decision
from src.assemble import assemble_output
from src.export import compute_feature_mean, write_excel

cfg = get_vertical(VERTICAL)
print(f"Vertical: {VERTICAL}")
print(f"  Decisión : {cfg['decision']}")
print(f"  Modelo   : {len(cfg['modelo']['features'])} features, clases con modelo: {list(cfg['modelo']['lgbm'])}")

## 1. Datos: BigQuery + insumos locales

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="acpe-dev-uc-ml")

t0 = time.time()
df_clientes = fetch_clientes(VERTICAL, client)
print(f"✓ {len(df_clientes):,} clientes ({time.time() - t0:.0f}s)")

t0 = time.time()
df_puntos = fetch_puntos(client)
print(f"✓ {len(df_puntos):,} puntos comerciales ({time.time() - t0:.0f}s)")

In [ ]:
locales = pd.read_csv(RUTA_LOCALES, sep=";", encoding="utf-8-sig")

t0 = time.time()
equifax_sum = prepare_equifax_sum(pd.read_csv(RUTA_EQUIFAX), pd.read_csv(RUTA_URBANICIDAD))
print(f"✓ equifax_sum: {len(equifax_sum):,} manzanas ({time.time() - t0:.0f}s)")

## 2-3. Features geográficas + Equifax

In [ ]:
df_features, info = agregar_features_cliente(df_clientes, df_puntos, locales, equifax_sum)
print(info)

df_features = df_features.rename(columns={"cod_cliente_dex": "Cod Cliente DEX"})
df_features["cluster"] = None  # placeholder si algún consumidor externo lo espera; se descarta al exportar

## 4. Modelo (LightGBM por clase, predicción cruzada, SHAP)

In [ ]:
m = cfg["modelo"]
df_modelo = df_features.copy()
if m["target_map"]:
    df_modelo["segmento_trx"] = df_modelo["segmento_trx"].map(m["target_map"])

t0 = time.time()
resultado = fit_potential(
    df_modelo, m["features"], m["categoricas"], cfg["class_order"], m["lgbm"],
    id_col="Cod Cliente DEX", cluster_col=m["cluster_col"], split_valid=m["split_validacion"],
    optuna_trials=m["optuna"]["n_trials"] if m["optuna"]["activo"] else None,
)
print(f"✓ {len(resultado.estimates):,} estimaciones ({time.time() - t0:.0f}s)")
log_metrics(VERTICAL, resultado, RUTA_METRICAS)
validation_table(resultado)

## 5. Decisión (Medium/High, flag_potential)

In [ ]:
d = cfg["decision"]
estimaciones = apply_decision(resultado.estimates, d["regla"], cfg["class_order"], d["high"], d["medium"])
print(f"Potenciales: {int(estimaciones['flag_potential'].sum()):,} de {len(estimaciones):,}")

## 6. Ensamblado, Feature mean y Excel

In [ ]:
salida = assemble_output(estimaciones, df_features, m["features"], id_col="Cod Cliente DEX")

fm = cfg["feature_mean"]
feature_mean = compute_feature_mean(
    salida, cfg["class_order"], fm["rmv_feats"], fm["modo"],
    features=[f for f in m["features"] if f not in set(fm["rmv_feats"])] if fm["modo"] == "features" else None,
    reference_df=df_features,  # llena la media de la clase alta (sin filas propias en `salida`)
)

diccionario = pd.read_csv(Path("..") / "configs" / "diccionario_encuesta.csv")

e = cfg["export"]
fecha = time.strftime("%Y%m%d")
ruta = Path(RUTA_SALIDA) / f"{fecha}_{cfg['salida']}.xlsx"
ruta.parent.mkdir(parents=True, exist_ok=True)
write_excel(
    str(ruta),
    {e["hoja_principal"]: salida, "Diccionario Encuesta": diccionario, "Feature mean": feature_mean},
    hojas_con_formato=[e["hoja_principal"]] if e["con_formato"] else [],
)
print(f"✓ Guardado en {ruta}")

## Correr todas las verticales de cliente

Ejecuta esta celda en vez de las anteriores para correrlas todas en un solo llamado (usa `run_vertical` definida abajo, que repite exactamente los pasos 1 a 6).

In [ ]:
def run_vertical(vertical: str, client, locales: pd.DataFrame, equifax_sum: pd.DataFrame,
                 diccionario: pd.DataFrame, ruta_salida: str = RUTA_SALIDA) -> Path:
    """Repite los pasos 1 a 6 de este notebook para `vertical`. Devuelve la ruta del Excel."""
    cfg = get_vertical(vertical)
    if cfg["granularidad"] != "cliente":
        raise ValueError(f"'{vertical}' no es de granularidad cliente; usa el orquestador de PM.")
    m, d, e = cfg["modelo"], cfg["decision"], cfg["export"]

    df_clientes = fetch_clientes(vertical, client)
    df_puntos = fetch_puntos(client)
    df_features, info = agregar_features_cliente(df_clientes, df_puntos, locales, equifax_sum)
    print(f"[{vertical}] {info}")
    df_features = df_features.rename(columns={"cod_cliente_dex": "Cod Cliente DEX"})

    df_modelo = df_features.copy()
    if m["target_map"]:
        df_modelo["segmento_trx"] = df_modelo["segmento_trx"].map(m["target_map"])
    resultado = fit_potential(
        df_modelo, m["features"], m["categoricas"], cfg["class_order"], m["lgbm"],
        id_col="Cod Cliente DEX", cluster_col=m["cluster_col"], split_valid=m["split_validacion"],
        optuna_trials=m["optuna"]["n_trials"] if m["optuna"]["activo"] else None,
    )
    log_metrics(vertical, resultado, Path(ruta_salida) / "metricas_modelos.csv")
    estimaciones = apply_decision(resultado.estimates, d["regla"], cfg["class_order"], d["high"], d["medium"])
    salida = assemble_output(estimaciones, df_features, m["features"], id_col="Cod Cliente DEX")

    fm = cfg["feature_mean"]
    feature_mean = compute_feature_mean(
        salida, cfg["class_order"], fm["rmv_feats"], fm["modo"],
        features=[f for f in m["features"] if f not in set(fm["rmv_feats"])] if fm["modo"] == "features" else None,
        reference_df=df_features,
    )

    fecha = time.strftime("%Y%m%d")
    ruta = Path(ruta_salida) / f"{fecha}_{cfg['salida']}.xlsx"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    write_excel(
        str(ruta),
        {e["hoja_principal"]: salida, "Diccionario Encuesta": diccionario, "Feature mean": feature_mean},
        hojas_con_formato=[e["hoja_principal"]] if e["con_formato"] else [],
    )
    print(f"✓ [{vertical}] {int(estimaciones['flag_potential'].sum()):,} potenciales de {len(estimaciones):,} — {ruta}")
    return ruta


VERTICALES_CLIENTE = ["lavanderias", "hoteles", "panaderia", "gastro", "bodegas"]

# equifax_sum y locales ya cargados arriba; si corres solo esta sección, descomenta:
# locales = pd.read_csv(RUTA_LOCALES, sep=";", encoding="utf-8-sig")
# equifax_sum = prepare_equifax_sum(pd.read_csv(RUTA_EQUIFAX), pd.read_csv(RUTA_URBANICIDAD))
# diccionario = pd.read_csv(Path("..") / "configs" / "diccionario_encuesta.csv")

with warnings.catch_warnings():
    warnings.simplefilter("default")  # que se vean los avisos de src.model (clases fuera de class_order, etc.)
    rutas = {v: run_vertical(v, client, locales, equifax_sum, diccionario) for v in VERTICALES_CLIENTE}